# Build Your Own Transformer

Companion notebook to the [Understanding Transformers](https://yattas.com/tutorials/transformers/) series. The lessons cover the same material in the order that's easiest to understand; this notebook builds it in the order you'd actually write it, one section leading into the next, so the tokenizer, embeddings, and trained weights carry forward from each section to the next.

Runs entirely on Colab's free CPU tier under your own Google account — nothing here needs a GPU, and training finishes in well under a minute.

**How this notebook uses PyTorch.** Every piece of the architecture — attention, multi-head attention, the feedforward network, layer normalization — is written out by hand, the same formulas as the lessons' NumPy examples. The one thing handed to PyTorch is backpropagation: once the forward pass and loss are computed, `.backward()` works out every gradient automatically, instead of hand-deriving the calculus for each piece. No `nn.Linear`, `nn.MultiheadAttention`, or `nn.TransformerEncoder` is used anywhere — those would build the model *for* you, which defeats the point of this notebook.

Use Colab's outline (View → Table of contents) to jump between sections.


## 1. Tokenization & Vocabulary

**Builds on:** [Lesson 6](https://yattas.com/tutorials/transformers/06-tokens-embeddings-position/)

Real text, not the four-word toy vocabulary from the lesson page. The tokenizer here works at the character level: every distinct character in the corpus becomes one vocabulary entry, so tokenizing is just looking up each character's ID and detokenizing is looking the ID back up. It's the simplest tokenizer that could work, and simple enough to build in a few lines rather than import.


In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# Runs on GPU automatically if Colab's runtime has one, CPU otherwise --
# every tensor below is created on (or moved to) this one device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# A real piece of text -- long enough to have repeated structure for the
# model to actually learn from, still short enough to train in seconds.
corpus = """The cat sat on the mat. The cat sat on the roof. The cat did not sit on
the moon, because the moon has no roof and no mat, only dust and quiet.
Every night the cat climbed to the roof and watched the moon instead. The
mat stayed on the floor. The roof stayed under the moon. The cat sat, and
sat, and sat, until the mat missed the cat and the cat missed the mat, and
the moon, who missed nothing, kept its slow watch over the roof, the mat,
and the cat who loved them both. Spring came, and the garden grew loud
with bees. The cat ignored the bees and kept its post on the roof, one ear
turned toward the mat below. Sometimes a bird would land near the door,
and the cat would freeze, tail flat, eyes wide, before deciding the bird
was not worth the climb down. The moon did not care about bees or birds.
The moon only cared about the roof, the mat, and the quiet that settled
over the house each night. Summer brought heat that pressed against the
windows. The cat moved from the roof to the shade under the porch, then
back to the mat when the shade grew too warm. Dust gathered in the corners
of the porch, the same dust that had always gathered there, patient and
grey. The cat sneezed once at the dust and never again. The moon rose
later in summer, but it still found the roof exactly where it had left it,
still found the mat, still found the cat curled against the cool tile by
the door. Autumn scattered leaves across the garden, and the wind pushed
them under the door and onto the mat. The cat batted at a leaf, lost
interest, and returned to watching the roof from below. The roof held its
shape against the wind, the way it always had, the way the cat trusted it
always would. At night the moon rose thin and pale, half hidden by clouds,
and the cat sat by the window and waited for it anyway. Winter came last,
and the mat grew cold under the cat's paws. The cat pressed closer to the
door where warmth leaked out from inside, and dreamed of the roof in
summer, of dust in sunlight, of the moon full and bright over everything
it loved. The roof wore a thin layer of frost. The mat stiffened in the
cold. And still, each night, the moon returned to keep its watch, and the
cat, half asleep, kept its own watch beside it, the two of them quiet
together under a sky that never quite went dark. The cat never explained
why it loved the roof so much, and the moon never once asked.""".replace("\n", " ")

# Vocabulary = every distinct character that appears, sorted for a stable order.
chars = sorted(set(corpus))
vocab_size = len(chars)
char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}

def encode(text):
    return [char_to_id[ch] for ch in text]

def decode(ids):
    return "".join(id_to_char[i] for i in ids)

# The whole corpus, tokenized once, reused by every later section.
data = torch.tensor(encode(corpus), dtype=torch.long, device=device)

print(f"vocab_size = {vocab_size}, corpus length = {len(corpus)} characters")
print("round trip check:", decode(encode("the cat")) == "the cat")


## 2. Embeddings & Positional Encoding

**Builds on:** [Lesson 6](https://yattas.com/tutorials/transformers/06-tokens-embeddings-position/)

Two lookup tables, both learned: one row per vocabulary entry (what a character *is*) and one row per position (*where* it sits in the sequence). Adding the two together gives each character a starting vector that encodes both — the same idea as the lesson's toy example, just with a real vocabulary and a longer sequence.


In [ ]:
d_model = 32       # size of every token's vector throughout the model
block_size = 48     # how many characters of context the model reads at once

# One learned row per vocabulary entry -- looking up an ID retrieves its row,
# exactly like the lesson's embedding_table[ids].
token_emb_table = torch.nn.Parameter(torch.randn(vocab_size, d_model, device=device) * 0.02)

# One learned row per position, 0 .. block_size-1.
pos_emb_table = torch.nn.Parameter(torch.randn(block_size, d_model, device=device) * 0.02)

# Try it on the first block_size characters of the corpus.
sample_ids = data[:block_size]
tok_emb = token_emb_table[sample_ids]
x = tok_emb + pos_emb_table[: len(sample_ids)]

print("sample text:", decode(sample_ids.tolist()))
print("embedded shape (sequence_length, d_model):", x.shape)


## 3. Self-Attention From Scratch

**Builds on:** Lessons [7](https://yattas.com/tutorials/transformers/07-why-attention/)-[8](https://yattas.com/tutorials/transformers/08-self-attention/)

The same steps as the lesson's worked example -- project to Q/K/V, score, scale, mask, normalize, mix -- run over a real chunk of the corpus instead of three toy tokens. The causal mask is new: it wasn't needed for a single static example, but it's essential once a real sequence is read left to right, so a token can never attend to ones that haven't happened yet.


In [ ]:
# Query/Key/Value projections -- plain matrices, no nn.Linear.
Wq = torch.nn.Parameter(torch.randn(d_model, d_model, device=device) * 0.02)
Wk = torch.nn.Parameter(torch.randn(d_model, d_model, device=device) * 0.02)
Wv = torch.nn.Parameter(torch.randn(d_model, d_model, device=device) * 0.02)

def self_attention(x, Wq, Wk, Wv):
    T = x.shape[0]   # sequence length
    Q, K, V = x @ Wq, x @ Wk, x @ Wv

    scores = Q @ K.T / math.sqrt(Q.shape[-1])          # scaled QK^T (Part 8)

    # Causal mask: position i may only look at positions 0..i.
    # Future positions get -inf so softmax turns them into exactly zero weight.
    causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
    scores = scores.masked_fill(causal_mask, float("-inf"))

    weights = F.softmax(scores, dim=-1)                # attention weights
    return weights @ V                                 # weighted blend of Values

out = self_attention(x, Wq, Wk, Wv)
print("self-attention output shape:", out.shape)


## 4. Multi-Head Attention

**Builds on:** [Lesson 9](https://yattas.com/tutorials/transformers/09-multi-head-attention/)

`d_model` splits evenly across `num_heads` heads, each running the exact self-attention from Section 3 in its own smaller slice of the space, in parallel. Their outputs are concatenated back to the full width and passed through one more learned matrix, `W_o`, exactly as in the lesson.


In [ ]:
class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # One shared projection per Q/K/V, later split into heads --
        # equivalent to a separate smaller Wq/Wk/Wv per head, just batched.
        self.Wq = torch.nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.Wk = torch.nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.Wv = torch.nn.Parameter(torch.randn(d_model, d_model) * 0.02)
        self.Wo = torch.nn.Parameter(torch.randn(d_model, d_model) * 0.02)  # output projection

    def forward(self, x):
        T, _ = x.shape
        Q, K, V = x @ self.Wq, x @ self.Wk, x @ self.Wv

        # Split the last dimension into (num_heads, head_dim), then bring
        # num_heads to the front so each head is attended to independently.
        Q = Q.view(T, self.num_heads, self.head_dim).transpose(0, 1)
        K = K.view(T, self.num_heads, self.head_dim).transpose(0, 1)
        V = V.view(T, self.num_heads, self.head_dim).transpose(0, 1)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(causal_mask, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        heads_out = weights @ V                          # (num_heads, T, head_dim)

        # Concatenate the heads back into one d_model-wide vector per position.
        concat = heads_out.transpose(0, 1).reshape(T, self.num_heads * self.head_dim)
        return concat @ self.Wo

mha = MultiHeadSelfAttention(d_model, num_heads=4).to(device)
mha_out = mha(x)
print("multi-head output shape:", mha_out.shape)


## 5. Feedforward, Residuals & LayerNorm

**Builds on:** Lessons [5](https://yattas.com/tutorials/transformers/05-depth-residuals-normalization/), [11](https://yattas.com/tutorials/transformers/11-feedforward-networks/)

The feedforward network is the lesson's expand-activate-compress shape (`W2 · ReLU(W1 x + b1) + b2`), and layer normalization is the same mean/standard-deviation formula from Lesson 13. Neither sublayer replaces its input -- each one's output is *added back* through a residual connection, so the representation from Section 4 is preserved and only nudged, not overwritten.


In [ ]:
class FeedForward(torch.nn.Module):
    def __init__(self, d_model, hidden_mult=4):
        super().__init__()
        hidden = d_model * hidden_mult   # expand, same 4x ratio as the original transformer
        self.W1 = torch.nn.Parameter(torch.randn(hidden, d_model) * 0.02)
        self.b1 = torch.nn.Parameter(torch.zeros(hidden))
        self.W2 = torch.nn.Parameter(torch.randn(d_model, hidden) * 0.02)
        self.b2 = torch.nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        h = F.relu(x @ self.W1.T + self.b1)   # expand + activate
        return h @ self.W2.T + self.b2         # compress back to d_model

def layer_norm(x, eps=1e-5):
    # Normalize each position's vector to zero mean, unit variance.
    mean = x.mean(dim=-1, keepdim=True)
    std = x.std(dim=-1, keepdim=True)
    return (x - mean) / (std + eps)

ffn = FeedForward(d_model).to(device)

# Pre-norm residual form from Lesson 13: x + Sublayer(LayerNorm(x))
a = x + mha(layer_norm(x))
x_next = a + ffn(layer_norm(a))
print("after attention + FFN, both with residuals:", x_next.shape)


## 6. Assembling One Transformer Block

**Builds on:** [Lesson 13](https://yattas.com/tutorials/transformers/13-transformer-block/)

Multi-head attention and the feedforward network, each wrapped in its own residual connection and pre-applied layer norm, combine into one `TransformerBlock`: `a = x + Attention(LayerNorm(x))`, then `x_next = a + FFN(LayerNorm(a))` -- the same two-line formula from the lesson, now runnable on a real chunk of text.


In [ ]:
class TransformerBlock(torch.nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model)

    def forward(self, x):
        a = x + self.attn(layer_norm(x))
        return a + self.ffn(layer_norm(a))

block = TransformerBlock(d_model, num_heads=4).to(device)
block_out = block(x)
print("one transformer block, same shape in and out:", block_out.shape)


## 7. Stacking Blocks & the Output Head

**Builds on:** Lessons [10](https://yattas.com/tutorials/transformers/10-logits-softmax/), [12](https://yattas.com/tutorials/transformers/12-layer-by-layer/)

A full model is just this block repeated -- each one taking the previous one's output as its input, the representation deepening layer by layer (Lesson 12). After the last block, a final layer norm and one more learned matrix (`W_unembed`) project each position's vector to one score per vocabulary entry: the logits from Lesson 10, ready for softmax.


In [ ]:
class TinyTransformer(torch.nn.Module):
    def __init__(self, vocab_size, d_model, block_size, num_heads, num_layers):
        super().__init__()
        self.block_size = block_size
        self.token_emb_table = torch.nn.Parameter(torch.randn(vocab_size, d_model) * 0.02)
        self.pos_emb_table = torch.nn.Parameter(torch.randn(block_size, d_model) * 0.02)
        self.blocks = torch.nn.ModuleList(
            [TransformerBlock(d_model, num_heads) for _ in range(num_layers)]
        )
        self.W_unembed = torch.nn.Parameter(torch.randn(d_model, vocab_size) * 0.02)

    def forward(self, ids):
        T = ids.shape[0]
        x = self.token_emb_table[ids] + self.pos_emb_table[:T]
        for block in self.blocks:
            x = block(x)               # Parts 5, 12, 13 -- deepens layer by layer
        x = layer_norm(x)              # final norm before the output head
        logits = x @ self.W_unembed    # Part 10 -- one score per vocabulary entry
        return logits

model = TinyTransformer(vocab_size, d_model, block_size, num_heads=4, num_layers=2).to(device)

logits = model(sample_ids)
probs = F.softmax(logits, dim=-1)
print("logits shape (sequence_length, vocab_size):", logits.shape)
print("each position's probabilities sum to 1:",
      torch.allclose(probs.sum(-1), torch.ones(len(sample_ids), device=device), atol=1e-4))


## 8. Training Loop

**Builds on:** Lessons [3](https://yattas.com/tutorials/transformers/03-gradients-learning/), [15](https://yattas.com/tutorials/transformers/15-training-generation/)

Every position in a training sequence is trained to predict the character right after it, all at once, under the causal mask from Section 3 -- exactly Lesson 15's description of how a decoder-only model trains. The loss is the same cross-entropy formula from that lesson. Computing its gradient by hand is where the earlier lessons stop short; here, `loss.backward()` does it, and `torch.optim.Adam` applies the update -- a smarter, adaptive version of the plain `w ← w − η·∇L` rule from the cheat sheet -- so training reliably converges in well under a minute on this small corpus.


In [ ]:
def get_batch(data, block_size, batch_size):
    # Pick random starting points and slice out (input, target) pairs where
    # target is input shifted one character to the right -- "predict the next one".
    max_start = len(data) - block_size - 1
    starts = torch.randint(0, max_start, (batch_size,))
    xb = torch.stack([data[s : s + block_size] for s in starts])
    yb = torch.stack([data[s + 1 : s + block_size + 1] for s in starts])
    return xb, yb

def forward_batch(model, xb):
    # TinyTransformer.forward handles one sequence at a time (as built in
    # Section 7); looping over the batch keeps that same code unchanged
    # instead of rewriting it to juggle an extra batch dimension.
    return torch.stack([model(row) for row in xb])

# A fresh, untrained model -- Section 7's instance was only for checking shapes.
model = TinyTransformer(vocab_size, d_model, block_size, num_heads=4, num_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

batch_size = 16
steps = 1000

for step in range(steps):
    xb, yb = get_batch(data, block_size, batch_size)
    logits = forward_batch(model, xb)                    # (batch, T, vocab_size)

    # Cross-entropy over every position in every sequence at once (Part 15).
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))

    optimizer.zero_grad()
    loss.backward()     # <- the one line that replaces hand-derived calculus
    optimizer.step()    # <- the one line that replaces w -= lr * grad by hand

    if step % 50 == 0 or step == steps - 1:
        print(f"step {step:4d}: loss {loss.item():.3f}")


## 9. Generation & Sampling

**Builds on:** Lessons [14](https://yattas.com/tutorials/transformers/14-kv-cache/)-[15](https://yattas.com/tutorials/transformers/15-training-generation/)

The same loop from Lesson 15: read the context, get a distribution over the next character, pick one, append it, repeat. Temperature, top-k, and top-p all shape *which* character gets picked, exactly as described there. Every step recomputes the whole context from scratch rather than reusing a KV cache (Lesson 14) -- at this size the difference is invisible, but it's the first thing worth adding if this model is ever scaled up.


In [ ]:
def generate(model, prompt_ids, num_new_tokens, temperature=1.0, top_k=None, top_p=None):
    ids = list(prompt_ids)
    for _ in range(num_new_tokens):
        # Only the last block_size characters fit in the model's context window.
        context = torch.tensor(ids[-block_size:], dtype=torch.long, device=device)
        logits = model(context)
        next_logits = logits[-1] / temperature   # temperature: sharpen (<1) or flatten (>1)

        if top_k is not None:
            # Keep only the k highest-scoring characters, -inf out the rest.
            values, indices = torch.topk(next_logits, top_k)
            filtered = torch.full_like(next_logits, float("-inf"))
            filtered[indices] = values
            next_logits = filtered

        if top_p is not None:
            # Keep the smallest set of characters whose probabilities add up to p
            # (nucleus sampling) -- a variable-size version of top-k.
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cumulative - sorted_probs) > top_p
            sorted_logits[cutoff] = float("-inf")
            next_logits = torch.full_like(next_logits, float("-inf"))
            next_logits[sorted_idx] = sorted_logits

        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()   # sample, don't just argmax
        ids.append(next_id)
    return decode(ids)

prompt = encode("The cat ")
with torch.no_grad():
    print("temperature 0.1, unfiltered: ", generate(model, prompt, 60, temperature=0.1))
    print("temperature 0.3, top_k 10:   ", generate(model, prompt, 60, temperature=0.3, top_k=10))
    print("temperature 0.5, top_p 0.9:  ", generate(model, prompt, 60, temperature=0.5, top_p=0.9))

# This corpus is tiny on purpose, so the model has largely memorized it --
# don't expect fluent new sentences. What to look for: real words and
# fragments from the training text, not random characters, and low
# temperature staying closer to the training text than high temperature.


## 10. Hardware Portability With JAX

**Builds on:** [Lesson 18](https://yattas.com/tutorials/transformers/18-hardware-accelerators/)

Every section above used PyTorch, which makes you decide where a tensor lives (`.to("cpu")` or `.to("cuda")`) -- a choice that's explicit throughout the code above. [JAX](https://github.com/google/jax), also from Google, takes a different approach: write ordinary NumPy-style array code, wrap the function in `jax.jit`, and JAX traces it once, compiles it with XLA, and reuses that compiled version on every later call -- unchanged whether XLA ends up targeting this notebook's CPU, a GPU, or a TPU. This section takes the same scaled dot-product attention from Section 3, writes it as exactly that kind of function, and prints the actual attention weights for one row so the causal mask's effect is visible, not just implied by a shape or a timing number.


In [ ]:
import jax
import jax.numpy as jnp
import time

print("JAX is running on:", jax.devices())

def self_attention_jax(x, Wq, Wk, Wv):
    # Identical math to Section 3's self_attention -- scaled QK^T, a causal
    # mask, softmax, then a weighted blend of V -- just written with
    # jax.numpy instead of torch. Nothing here is JAX-specific yet.
    T = x.shape[0]
    Q, K, V = x @ Wq, x @ Wk, x @ Wv

    scores = Q @ K.T / jnp.sqrt(Q.shape[-1])

    causal_mask = jnp.triu(jnp.ones((T, T)), k=1).astype(bool)
    scores = jnp.where(causal_mask, -jnp.inf, scores)

    weights = jax.nn.softmax(scores, axis=-1)   # returned too, just to look at
    return weights, weights @ V

key = jax.random.PRNGKey(0)
kx, kq, kk, kv = jax.random.split(key, 4)
x_demo = jax.random.normal(kx, (block_size, d_model)) * 0.02
Wq_demo = jax.random.normal(kq, (d_model, d_model)) * 0.02
Wk_demo = jax.random.normal(kk, (d_model, d_model)) * 0.02
Wv_demo = jax.random.normal(kv, (d_model, d_model)) * 0.02

# Run the plain function first -- no compilation, just Python calling it directly.
weights_eager, out_eager = self_attention_jax(x_demo, Wq_demo, Wk_demo, Wv_demo)

# The one line that makes it portable: wrap the plain function in jax.jit.
# The first call traces the function and compiles it with XLA; every call
# after that reuses the compiled version -- the same jitted function would
# run unchanged on a GPU or TPU runtime, no code here would need to change.
self_attention_jitted = jax.jit(self_attention_jax)
weights_jit, out_jit = self_attention_jitted(x_demo, Wq_demo, Wk_demo, Wv_demo)

# jax.jit only changes *how* the function runs, never *what* it computes --
# the eager and jitted calls above produce the exact same numbers.
print("eager and jitted outputs match exactly:", bool(jnp.allclose(out_eager, out_jit)))

# What the causal mask actually does: position 5 can attend to positions
# 0-5 (itself and everything before it), and is blocked from positions
# 6 and later, which haven't happened yet. Softmax turns "blocked" into
# exactly zero, so those weights simply don't show up below.
print("\nposition 5's attention weights over all", block_size, "positions:")
print(jnp.round(weights_jit[5], 3))
print("weights for positions 6+ are all zero:", bool(jnp.all(weights_jit[5, 6:] == 0)))
print("row sums to 1 (softmax over the positions it can see):", float(weights_jit[5].sum()))

# Timing: the *second* jitted call reuses the already-compiled program --
# no retracing, no recompilation, just the compiled version running again.
out_jit.block_until_ready()
start = time.time()
for _ in range(100):
    _, out_jit = self_attention_jitted(x_demo, Wq_demo, Wk_demo, Wv_demo)
out_jit.block_until_ready()
print(f"\n100 compiled calls: {time.time() - start:.4f}s")
